# Tutorial 01: Hello Tinker

Tinker is a remote GPU service for LLM training and inference. You write training loops in Python on your local machine; Tinker executes the heavy GPU operations (forward passes, backpropagation, sampling) on remote workers.

```
Your machine (CPU)                    Tinker Service (GPU)
+-----------------------+             +------------------------+
| Python training loop  |  -------->  | Forward/backward pass  |
| Data preparation      |  <--------  | Optimizer steps        |
| Evaluation logic      |             | Text generation        |
+-----------------------+             +------------------------+
```

You control the logic. Tinker runs the compute.

## Setup

Install the Tinker SDK and set your API key. Get a key from the [Tinker console](https://tinker-console.thinkingmachines.ai).

In [1]:
# !pip install tinker
import tinker
from tinker import types

## The client hierarchy

The entry point to Tinker is the **ServiceClient**. From it, you create specialized clients:

- **SamplingClient** -- generates text from a model (inference)
- **TrainingClient** -- runs forward/backward passes and optimizer steps (training)

Both talk to the same remote GPU workers. Let's start with the ServiceClient.

In [2]:
# Create a ServiceClient. This reads TINKER_API_KEY from your environment.
service_client = tinker.ServiceClient()

# Check what models are available
capabilities = service_client.get_server_capabilities()
print("Available models:")
for model in capabilities.supported_models:
    print(f"  - {model.model_name}")

Available models:
  - deepseek-ai/DeepSeek-V3.1
  - deepseek-ai/DeepSeek-V3.1-Base
  - deepseek-ai/DeepSeek-V3.1:peft:131072
  - moonshotai/Kimi-K2-Thinking
  - moonshotai/Kimi-K2-Thinking:peft:32768:tibo-v2
  - moonshotai/Kimi-K2-Thinking:peft:262144
  - moonshotai/Kimi-K2.5
  - moonshotai/Kimi-K2.5:peft:131072
  - meta-llama/Llama-3.1-70B
  - meta-llama/Llama-3.1-8B
  - meta-llama/Llama-3.1-8B-Instruct
  - meta-llama/Llama-3.2-1B
  - meta-llama/Llama-3.2-3B
  - meta-llama/Llama-3.3-70B-Instruct
  - nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  - nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16
  - nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16:peft:262144
  - Qwen/Qwen3-235B-A22B-Instruct-2507
  - Qwen/Qwen3-30B-A3B
  - Qwen/Qwen3-30B-A3B-Base
  - Qwen/Qwen3-30B-A3B-Instruct-2507
  - Qwen/Qwen3-32B
  - Qwen/Qwen3-4B-Instruct-2507
  - Qwen/Qwen3-8B
  - Qwen/Qwen3-8B-Base
  - Qwen/Qwen3-VL-235B-A22B-Instruct
  - Qwen/Qwen3-VL-30B-A3B-Instruct
  - Qwen/Qwen3.5-27B
  - Qwen/Qwen3.5-35B-A3B
  

## Sampling from a model

Let's create a **SamplingClient** to generate text. We will use `Qwen/Qwen3-4B-Instruct-2507`, a compact model that keeps costs low.

The sampling workflow is:
1. Create a `SamplingClient` with a base model name
2. Encode your prompt into tokens using the model's tokenizer
3. Call `sample()` with the prompt and sampling parameters
4. Decode the returned tokens back into text

In [3]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# Create a sampling client -- this connects to a remote GPU worker
sampling_client = service_client.create_sampling_client(base_model=MODEL_NAME)

# Get the tokenizer for encoding/decoding text
tokenizer = sampling_client.get_tokenizer()

/Users/yujia/Repos/tinker-cookbook/.claude/worktrees/quirky-discovering-lightning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Encode a prompt into tokens
prompt_text = "The three largest cities in the world by population are"
prompt = types.ModelInput.from_ints(tokenizer.encode(prompt_text))

# Sample a completion
params = types.SamplingParams(max_tokens=50, temperature=0.7, stop=["\n"])
future = sampling_client.sample(prompt=prompt, sampling_params=params, num_samples=1)

# .sample() returns a future immediately -- call .result() to wait for the GPU response
result = future.result()

# Decode and print
completion_tokens = result.sequences[0].tokens
print(prompt_text + tokenizer.decode(completion_tokens))

The three largest cities in the world by population are Tokyo, New York, and Mumbai. Tokyo has a population of 14 million, New York has 8 million, and Mumbai has 20 million. What is the average population of these three cities?




## Inspecting the response

The `sample()` call returns a `SampleResponse` containing a list of `SampledSequence` objects. Each sequence has:
- `tokens` -- the generated token IDs
- `logprobs` -- log probability of each generated token (if requested)
- `stop_reason` -- why generation stopped (e.g., hit max tokens, hit a stop string)

In [5]:
seq = result.sequences[0]

print(f"Stop reason:    {seq.stop_reason}")
print(f"Tokens generated: {len(seq.tokens)}")
print(f"Token IDs:      {seq.tokens[:10]}...")  # first 10
print(f"Log probs:      {seq.logprobs}")

Stop reason:    stop
Tokens generated: 43
Token IDs:      [26194, 11, 1532, 4261, 11, 323, 34712, 13, 26194, 702]...
Log probs:      [-0.5791736245155334, -0.004706376697868109, -2.2813668251037598, -0.0027554186526685953, -0.06658590584993362, -1.1920928244535389e-07, -1.0936973094940186, -0.015433523803949356, -2.7365012168884277, -0.14959144592285156, -1.2462809085845947, -0.0003152588615193963, -0.005192840471863747, -0.029316507279872894, -0.03357487916946411, -0.19611313939094543, -0.11535511165857315, -0.0122367599979043, -0.0019672818016260862, 0.0, -6.508615479106084e-05, -1.9407578706741333, -0.0024523441679775715, -0.11093226820230484, -0.000254241080256179, -1.311301275563892e-06, -1.1920928244535389e-07, -7.152531907195225e-06, -0.00022909401741344482, -0.6932312250137329, -0.00021002470748499036, 0.0, -0.00014435203047469258, -0.2887098491191864, -0.004086359404027462, -4.7205765440594405e-05, -1.29133141040802, -0.002534988336265087, -6.687417771900073e-05, -0.0194885246

You can also generate multiple samples at once by setting `num_samples`. Each sample is an independent completion from the same prompt.

In [6]:
# Generate 3 independent completions
result = sampling_client.sample(
    prompt=prompt,
    sampling_params=types.SamplingParams(max_tokens=50, temperature=0.9, stop=["\n"]),
    num_samples=3,
).result()

for i, seq in enumerate(result.sequences):
    text = tokenizer.decode(seq.tokens)
    print(f"Sample {i}: {text}")

Sample 0:  Tokyo, Delhi, and Shanghai. Tokyo has a population of 14 million, Delhi has a population of 11 million, and Shanghai has a population of 12 million. What is the total population of these three cities combined?

Sample 1: :

Sample 2:  Tokyo, Delhi, and Shanghai. Tokyo's population is 11 million more than Delhi's. Delhi's population is 4 million less than Shanghai's. If the total population of the three cities combined is 300 million, what is


## What about training?

So far we have only done inference. The real power of Tinker is **training** -- running forward/backward passes and optimizer steps on remote GPUs while you control the training loop locally.

The workflow looks like this:

1. Create a **TrainingClient** with `service_client.create_lora_training_client()`
2. Prepare training data as `Datum` objects (input tokens + loss targets)
3. Call `training_client.forward_backward()` to compute gradients
4. Call `training_client.optim_step()` to update weights
5. Save weights and create a **SamplingClient** to evaluate the trained model

We will walk through this in the next tutorial.

## Next steps

- **[Tutorial 02: First SFT](./02_first_sft.ipynb)** -- Train a model with supervised fine-tuning
- **[Getting Started Guide](https://docs.thinkingmachines.ai/training-sampling)** -- Full walkthrough of training and sampling
- **[Available Models](https://docs.thinkingmachines.ai/model-lineup)** -- All supported models and their characteristics